# Analysis for the $\sin$ benchmark

The datasets are created as follows.

1. `sin-noise-0`:
    Each $x_j$ is selected uniformly at random between $-\pi$ and $\pi$, $j = 1 \dots 100$.
    Then $y_j = \sin x_j$.
2. `sin-noise-1`:
    Same process as `sin-noise-0` but $y_j = \sin x_j + \eta_j$ where $\eta_j$ has a normal distribution with mean $0$ and standard deviation $0.1$.
3. `sin-noise-2`:
    Same process as `sin-noise-1` but the standard deviation is $0.01$.
4. `sin-wide-noise-k`:
    Same processes as `sin-noise-k` but with $x_j$ drawn uniformly between $-2\pi$ and $2\pi$, $j = 1 \dots 200$.


## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

These are the datasets to be fit.

In [ ]:
sn0 = pd.read_csv("datasets/sin-noise-0.csv")
sn1 = pd.read_csv("datasets/sin-noise-1.csv")
sn2 = pd.read_csv("datasets/sin-noise-2.csv")
sw0 = pd.read_csv("datasets/sin-wide-noise-0.csv")
sw1 = pd.read_csv("datasets/sin-wide-noise-1.csv")
sw2 = pd.read_csv("datasets/sin-wide-noise-2.csv")

In [ ]:
sns.scatterplot(data=sn0, x="x", y="y", color="blue", label=r"$\sigma= 0$")

In [ ]:
sns.scatterplot(data=sn2, x="x", y="y", color="green", label=r"$\sigma = 0.01$")

In [ ]:
sns.scatterplot(data=sn1, x="x", y="y", color="orange", label=r"$\sigma = 0.1$")

In [ ]:
sns.scatterplot(data=sw0, x="x", y="y", color="blue", label=r"$\sigma= 0$")

In [ ]:
sns.scatterplot(data=sw2, x="x", y="y", color="green", label=r"$\sigma = 0.01$")

In [ ]:
sns.scatterplot(data=sw1, x="x", y="y", color="orange", label=r"$\sigma = 0.1$")

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

full_report.loc[full_report.run_set=="SRB-2026-06-25-1715", "Lop"] = 0.0
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L4", "Lop"] = 1.0e-4
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L2", "Lop"] = 1.0e-2

# These have to be strings because they need to be exact categorical labels for plotting
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715", "Lopstr"] = r"$\lambda = 0$"
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L4", "Lopstr"] = r"$\lambda = 10^{-4}$"
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L2", "Lopstr"] = r"$\lambda = 10^{-2}$"

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(lambda e: au.parse_if_needed(e))
full_report["complexity"] = full_report.sympy.apply(lambda e: au.complexity(e))
full_report["sympy_defuzz"] = full_report.sympy.apply(lambda e: au.replace_near_integer(e))
full_report["complexity_defuzz"] = full_report.sympy_defuzz.apply(lambda e: au.complexity(e))

In [ ]:
full_report

## For reference

Symbolic regression is fitting noise if it gets MSE any lower than these.

In [ ]:
au.mse(np.sin(sn2.x), sn2.y)

In [ ]:
au.mse(np.sin(sn1.x), sn1.y)

In [ ]:
au.mse(np.sin(sw2.x), sw2.y)

In [ ]:
au.mse(np.sin(sw1.x), sw1.y)

## Analysis of results from the narrow datasets

### One period, zero noise

In [ ]:
results_sn0 = full_report[full_report.data_set == "sin-noise-0"].sort_values(by="mse")

In [ ]:
sns.histplot(data=results_sn0.mse, log_scale=True)

In [ ]:
sns.histplot(data=results_sn0.complexity)

With no noise, the function $\sin x$ is recovered every time, apart from fuzz.

In [ ]:
sns.histplot(data=results_sn0.complexity_defuzz)

### One period, low noise, very low $\lambda_{\mathrm{op}}$

In [ ]:
results_sn2 = full_report[full_report.data_set == "sin-noise-2"].sort_values(by="mse")

In [ ]:
results_sn2

In [ ]:
sns.histplot(data=results_sn2.mse, log_scale=True)

In [ ]:
sns.histplot(data=results_sn2.complexity)

In [ ]:
sns.histplot(data=results_sn2.complexity_defuzz)

In [ ]:
exprs_sn2 = results_sn2.sympy

In [ ]:
exprs_sn2

In [ ]:
exprs_sn2.apply(lambda e: au.replace_near_integer(e, tolerance=6.0e-3))

Exact recovery isn't too bad.
These are essentially correct, but it's hard to see.
There's a lot of cruft, and things like $-\cos(x + \pi/2)$, $\sin(x + \varepsilon)$, etc. instead of $\sin x$. 

Here's one that's almost correct

In [ ]:
sin_n2_ex = exprs_sn2.loc[76]
sin_n2_ex

In [ ]:
sin_n2_ex_y = au.apply_sym(sin_n2_ex, sn2.x)

In [ ]:
sns.scatterplot(x=sn2.x, y=sn2.y, color="blue", label="prediction")

### Two periods, low noise, very low $\lambda_{\mathrm{op}}$

In [ ]:
results_sw2 = full_report[full_report.data_set == "sin-wide-noise-2"].sort_values(by="mse")

In [ ]:
results_sw2

In [ ]:
sns.histplot(data=results_sw2.mse, log_scale=True)

In [ ]:
exprs_sw2 = results_sw2.sympy

These are all essentially correct, just with a lot of fuzz and cruft.

### One period, high noise, very low $\lambda_{\mathrm{op}}$

In [ ]:
results_sn1 = full_report[full_report.data_set == "sin-noise-1"].sort_values(by="mse")

In [ ]:
results_sn1

In [ ]:
sns.histplot(data=results_sn1.mse, log_scale=True)

In [ ]:
exprs_sn1 = results_sn1.sympy

In [ ]:
exprs_sn1

In [ ]:
exprs_sn1.apply(lambda e: au.replace_near_integer(e, tolerance=6.0e-2))

Many of these aren't bad, but the cruft is significant.

### Two period, high noise, very low $\lambda_{\mathrm{op}}$

In [ ]:
results_sw1 = full_report[full_report.data_set == "sin-wide-noise-1"].sort_values(by="mse")

In [ ]:
results_sw1

In [ ]:
sns.histplot(data=results_sw1.mse, log_scale=True)

In [ ]:
exprs_sw1 = results_sw1.sympy

In [ ]:
exprs_sw1

In [ ]:
exprs_sn1.apply(lambda e: au.replace_near_integer(e, tolerance=6.0e-3))

## Combination analysis

Jessamine is able to more or less recover $\sin x$ in all cases, but the amount of fuzz and cruft varies considerably.

In [ ]:
sns.displot(full_report, x="mse", col="data_set", row="run_set",
            log_scale=True)

In [ ]:
sns.displot(full_report, x="complexity", col="data_set", row="run_set",
            log_scale=True)

In [ ]:
sns.displot(full_report, x="complexity_defuzz", col="data_set", row="run_set",
            log_scale=True)

### One period, low noise, various values of $\lambda_{\mathrm{op}}$

In [ ]:
results_sn2 = full_report[full_report.data_set == "sin-noise-2"].sort_values(by="mse")

In [ ]:
results_sn2

In [ ]:
results_sn2_low_L = results_sn2[results_sn2.Lop == 0.0]
results_sn2_mid_L = results_sn2[results_sn2.Lop == 1.0e-4]
results_sn2_high_L = results_sn2[results_sn2.Lop == 1.0e-2]

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn2_low_L.mse,
    "Mid": results_sn2_mid_L.mse,
    "High": results_sn2_high_L.mse
}),
log_scale=True, multiple="dodge", palette="pastel")

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn2_low_L.complexity,
    "Mid": results_sn2_mid_L.complexity,
    "High": results_sn2_high_L.complexity
}),
bins=range(0,200,10), multiple="dodge", palette="pastel")

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn2_low_L.complexity_defuzz,
    "Mid": results_sn2_mid_L.complexity_defuzz,
    "High": results_sn2_high_L.complexity_defuzz
}),
bins=range(0,200,10), multiple="dodge", palette="pastel")

So for low noise, increasing $\lambda_{{\mathrm{op}}}$ definitely lowers the complexity without increasing the MSE past the known correct value.

### One period, high noise, various values of $\lambda_{\mathrm{op}}$

In [ ]:
results_sn1 = full_report[full_report.data_set == "sin-noise-1"].sort_values(by="Lop")

In [ ]:
results_sn1

In [ ]:
results_sn1_low_L = results_sn1[results_sn1.Lop == 0.0]
results_sn1_mid_L = results_sn1[results_sn1.Lop == 1.0e-4]
results_sn1_high_L = results_sn1[results_sn1.Lop == 1.0e-2]

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn1_low_L.mse,
    "Mid": results_sn1_mid_L.mse,
    "High": results_sn1_high_L.mse
}),
log_scale=True, multiple="dodge", palette="pastel")

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn1_low_L.complexity,
    "Mid": results_sn1_mid_L.complexity,
    "High": results_sn1_high_L.complexity
}),
bins=range(0,200,10), multiple="dodge", palette="pastel")

In [ ]:
sns.histplot(data=pd.DataFrame({
    "Low": results_sn1_low_L.complexity_defuzz,
    "Mid": results_sn1_mid_L.complexity_defuzz,
    "High": results_sn1_high_L.complexity_defuzz
}),
bins=range(0,200,10), multiple="dodge", palette="pastel")

In [ ]:
sn1_df = full_report[full_report.data_set == "sin-noise-1"].sort_values(by="Lop")
sn1_df

In [ ]:
sns.scatterplot(
    data = sn1_df,
    x="complexity",
    y="mse",
    style="Lopstr",
    hue="Lopstr",
)

In [ ]:
sns.displot(
    data = sn1_df,
    x="complexity",
    y="mse",
    hue="Lopstr",
    kind="hist",
    bins=20
)

So for low noise, increasing $\lambda_{{\mathrm{op}}}$ definitely lowers the complexity without increasing the MSE past the known correct value.
The middle value of $\lambda_{{\mathrm{op}}}$ helps some, but you really need the high value to get good results.

In [ ]:
results_sn1_high_L.sympy_defuzz

These are all basically correct with low cruft.

So the story here is that if there's noise, the complexity measure has to be of approximately the same order of magnitude as the noise, otherwise, it will add a lot of cruft trying to fit the noise.
The effect is very noticeable.
Which means that you have to estimate the noise before running symbolic regression.